In [ ]:
"""
Simplified Climate Report Analyzer with Step-by-Step Pipeline
- Explicit steps: extract → translate → analyze
- Each step can be run independently
- Strategic caching only for slow operations
"""

import json
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from datetime import datetime
import pdfplumber
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from tqdm.auto import tqdm
import langid
import random



class ClimateReportAnalyzer:
    MIN_CHARS = 200
    MAX_CHARS = 2000
    MAX_TOKENS = MAX_CHARS//4
    BATCH_SIZE = 48

    def __init__(self, cache_dir="cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        self.device = 0 if torch.cuda.is_available() else -1
        self.models = {}
        self.translators = {}

        self.supported_languages = {
            'de': 'Helsinki-NLP/opus-mt-de-en',
            'fr': 'Helsinki-NLP/opus-mt-fr-en',
            'es': 'Helsinki-NLP/opus-mt-es-en',
            'it': 'Helsinki-NLP/opus-mt-it-en',
            'pt': 'Helsinki-NLP/opus-mt-tc-big-pt-en',
            'nl': 'Helsinki-NLP/opus-mt-nl-en',
            'pl': 'Helsinki-NLP/opus-mt-pl-en',
            'sv': 'Helsinki-NLP/opus-mt-sv-en',
            'da': 'Helsinki-NLP/opus-mt-da-en',
            'fi': 'Helsinki-NLP/opus-mt-fi-en',
            'no': 'Helsinki-NLP/opus-mt-no-en',
            'ru': 'Helsinki-NLP/opus-mt-ru-en',
            'zh': 'Helsinki-NLP/opus-mt-zh-en',
            'ja': 'Helsinki-NLP/opus-mt-ja-en',
            'ko': 'Helsinki-NLP/opus-mt-ko-en',
        }

    # ========================================================================
    # STEP 1: EXTRACT PDF TEXT
    # ========================================================================

    def extract_pdf(self, pdf_path: str) -> Dict:
        """
        Extract text from PDF and chunk it
        Returns: Dict with text, language, and chunks
        """
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_extracted.json"

        # Check cache
        if cache_file.exists():
            print(f"✓ Loading cached extraction for {pdf_stem}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)

        print(f"\n{'='*80}")
        print(f"STEP 1: EXTRACTING PDF")
        print(f"{'='*80}\n")

        # Extract text
        text = self._extract_text_from_pdf(pdf_path)
        print(f"✓ Extracted {len(text):,} characters")

        # Detect language
        lang = self._detect_language(text)

        # Chunk text
        chunks = self._chunk_text(text)
        print(f"✓ Created {len(chunks)} chunks")

        # Save
        result = {
            "pdf_path": str(pdf_path),
            "language": lang,
            "num_chunks": len(chunks),
            "chunks": chunks,
            "extracted_at": str(datetime.now())
        }

        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"✓ Cached to {cache_file.name}\n")
        return result

    def _extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract raw text from PDF pages"""
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in tqdm(pdf.pages, desc="Reading pages"):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text

    def _detect_language(self, text: str) -> str:
        """Detect language using langid"""
        try:
            sample = text[:5000]
            lang, _ = langid.classify(sample)
            print(f"✓ Language detected: {lang}")
            return lang
        except Exception as e:
            print(f"⚠ Language detection failed: {e}. Defaulting to 'en'")
            return 'en'

    def _chunk_text(self, text: str) -> List[str]:
        """Split text into chunks by paragraphs"""

        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para_size = len(para)
            current_size = len(current_chunk)

            # Large paragraph: split by sentences
            if para_size > self.MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = ""

                sentences = para.split('. ')
                for sent in sentences:
                    if current_size + len(sent) < self.MAX_CHARS:
                        current_chunk += sent + ". "
                        current_size = len(current_chunk)
                    else:
                        if current_chunk:
                            chunks.append(current_chunk.strip())
                        current_chunk = sent + ". "
                        current_size = len(current_chunk)

            # Would exceed limit: save current and start new
            elif current_size + para_size > self.MAX_CHARS:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = para

            # Small paragraph: accumulate
            else:
                current_chunk += ("\n\n" if current_chunk else "") + para

        if current_chunk:
            chunks.append(current_chunk.strip())

        # Filter very short chunks
        return [c for c in chunks if len(c) > self.MIN_CHARS]

    # ========================================================================
    # STEP 2: TRANSLATE (OPTIONAL)
    # ========================================================================

    def translate_pdf(self, pdf_path: str) -> Dict:
        """
        Translate extracted chunks to English
        Returns: Dict with original and translated chunks
        """
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_translated.json"

        # Check cache
        if cache_file.exists():
            print(f"✓ Loading cached translation for {pdf_stem}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)

        print(f"\n{'='*80}")
        print(f"STEP 2: TRANSLATING")
        print(f"{'='*80}\n")

        # Load extracted data
        extracted = self.extract_pdf(pdf_path)
        lang = extracted['language']
        chunks = extracted['chunks']

        # Check if translation needed
        if lang == 'en':
            print(f"✓ Already in English, no translation needed")
            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": False,
                "chunk_pairs": [
                    {"original": c, "translated": c} for c in chunks
                ],
                "translated_at": str(datetime.now())
            }
        elif lang not in self.supported_languages:
            print(f"⚠ Translation not supported for language: {lang}")
            print(f"  Supported: {list(self.supported_languages.keys())}")
            print(f"  Proceeding with original text\n")
            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": False,
                "unsupported": True,
                "chunk_pairs": [
                    {"original": c, "translated": c} for c in chunks
                ],
                "translated_at": str(datetime.now())
            }
        else:
            # Translate
            translated_chunks = self._translate_chunks(chunks, lang)
            print(f"✓ Translated {len(translated_chunks)} chunks")

            # Verify translation worked
            detected = langid.classify(translated_chunks[0][:500])[0]
            print(f"✓ First chunk detected as: {detected}\n")

            result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": True,
                "chunk_pairs": [
                    {"original": orig, "translated": trans}
                    for orig, trans in zip(chunks, translated_chunks)
                ],
                "translated_at": str(datetime.now())
            }

        # Save
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"✓ Cached to {cache_file.name}\n")
        return result

    def _load_translator(self, lang: str):
        """Load translation model for given language"""
        if lang not in self.translators:
            print(f"Loading translator {lang}→en...")
            model_name = self.supported_languages[lang]
            self.translators[lang] = pipeline(
                "translation",
                model=model_name,
                device=self.device,
                max_length=600  # Allow longer outputs
            )
        return self.translators[lang]

    def _translate_chunks(self, chunks: List[str], lang: str,
                         batch_size=32) -> List[str]:
        """Translate chunks to English"""
        translator = self._load_translator(lang)
        translated = []

        print(f"Translating {len(chunks)} chunks from {lang}→en...")

        for i in tqdm(range(0, len(chunks), batch_size), desc="Translating"):
            batch = chunks[i:i+batch_size]

            try:
                # Truncate to XX (safer for 512 token limit with margin)
                truncated = [chunk[:self.MAX_CHARS] for chunk in batch]

                # Call translator - it returns a list of dicts
                results = translator(
                    truncated,
                    batch_size=batch_size,
                    truncation=True,
                    max_length=512  # Match model's limit
                )

                # Extract translation_text from each result
                # Results is a list of lists: [[{'translation_text': '...'}], ...]
                for result in results:
                    if isinstance(result, list) and len(result) > 0:
                        # Most common case: [{'translation_text': 'translated text'}]
                        text = result[0].get('translation_text', '')
                    elif isinstance(result, dict):
                        # Fallback: {'translation_text': 'translated text'}
                        text = result.get('translation_text', '')
                    else:
                        # Should never happen, but handle gracefully
                        print(f"\n⚠ Unexpected result format: {type(result)}")
                        text = ''

                    # Validate translation actually happened
                    if not text or text == truncated[len(translated) % len(truncated)]:
                        print(f"\n⚠ Translation failed for chunk {len(translated)}, using original")
                        text = batch[len(translated) % len(batch)]

                    translated.append(text)

            except Exception as e:
                print(f"\n⚠ Translation error in batch {i//batch_size}: {type(e).__name__}: {e}")
                print(f"   Batch size: {len(batch)}, Longest chunk: {max(len(c) for c in batch)} chars")
                # Fallback to original for this batch
                translated.extend(batch)

        return translated

    # ========================================================================
    # INSPECTION HELPERS
    # ========================================================================

    def inspect_extraction(self, pdf_path: str, n_samples=3):
        """Show extracted chunks"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_extracted.json"

        if not cache_file.exists():
            print(f"No extracted data found. Run extract_pdf() first.")
            return

        with open(cache_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print(f"\n{'='*80}")
        print(f"EXTRACTION INSPECTION: {Path(pdf_path).name}")
        print(f"Language: {data['language']}")
        print(f"Total chunks: {data['num_chunks']}")
        print(f"{'='*80}\n")

        chunks = data['chunks']
        samples = random.sample(chunks, min(n_samples, len(chunks)))

        for i, chunk in enumerate(samples, 1):
            print(f"{'─'*80}")
            print(f"Chunk {i} ({len(chunk)} chars)")
            print(f"{'─'*80}")
            print(f"{chunk[:400]}{'...' if len(chunk) > 400 else ''}\n")

    def inspect_translation(self, pdf_path: str, n_samples=3):
        """Show original vs translated chunks side-by-side"""
        pdf_stem = Path(pdf_path).stem
        cache_file = self.cache_dir / f"{pdf_stem}_translated.json"

        if not cache_file.exists():
            print(f"No translation data found. Run translate_pdf() first.")
            return

        with open(cache_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        lang = data['language']
        pairs = data['chunk_pairs']
        was_translated = data.get('translated', False)

        print(f"\n{'='*80}")
        print(f"TRANSLATION INSPECTION: {Path(pdf_path).name}")
        print(f"Language: {lang}")
        print(f"Translated: {was_translated}")
        print(f"Total chunks: {len(pairs)}")
        print(f"{'='*80}\n")

        samples = random.sample(pairs, min(n_samples, len(pairs)))

        for i, pair in enumerate(samples, 1):
            orig = pair['original']
            trans = pair['translated']

            print(f"{'─'*80}")
            print(f"Sample {i}")
            print(f"{'─'*80}")
            print(f"\nORIGINAL ({lang}):")
            print(f"{orig[:400]}{'...' if len(orig) > 400 else ''}")

            if orig != trans:  # Only show if actually different
                print(f"\nTRANSLATED (en):")
                print(f"{trans[:400]}{'...' if len(trans) > 400 else ''}")

            print()

    # ========================================================================
    # STEP 3: ANALYSIS
    # ========================================================================

    def load_model(self, model_name: str, task_name: str):
        """Load and cache a model"""
        if task_name not in self.models:
            print(f"Loading {task_name} model...")
            model = AutoModelForSequenceClassification.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.models[task_name] = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                device=self.device
            )
        return self.models[task_name]

    def get_chunks_for_analysis(self, pdf_path: str,
                                use_translated: bool = True) -> List[str]:
        """
        Get chunks ready for analysis

        Args:
            pdf_path: Path to PDF
            use_translated: Use translated chunks if available (default True)

        Returns:
            List of text chunks (translated if available and requested)
        """
        pdf_stem = Path(pdf_path).stem
        trans_cache = self.cache_dir / f"{pdf_stem}_translated.json"
        extract_cache = self.cache_dir / f"{pdf_stem}_extracted.json"

        # Try translated first
        if use_translated and trans_cache.exists():
            with open(trans_cache, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return [pair['translated'] for pair in data['chunk_pairs']]

        # Fall back to extracted
        if extract_cache.exists():
            with open(extract_cache, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data['chunks']

        # Nothing cached, extract now
        extracted = self.extract_pdf(pdf_path)
        return extracted['chunks']

    def filter_climate_chunks(self, chunks: List[str], batch_size=None) -> List[Dict]:
        """Filter for climate-related chunks"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector",
            "detector"
        )

        print(f"\nFiltering {len(chunks)} chunks for climate content...")
        climate_chunks = []

        for i in tqdm(range(0, len(chunks), batch_size), desc="Detecting"):
            batch = chunks[i:i+batch_size]

            try:
                results = detector(batch, truncation=True, max_length=512,
                                 batch_size=batch_size)

                for chunk, result in zip(batch, results):
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'text': chunk,
                            'detector_score': result['score']
                        })
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        print(f"✓ Kept {len(climate_chunks)} climate chunks "
              f"({len(climate_chunks)/len(chunks)*100:.1f}%)")
        return climate_chunks




In [17]:
analyzer = ClimateReportAnalyzer()
pdf_path = "data/reports/Dillinger/012_2015_annual_report.pdf"

# STEP 1: Extract PDF (always do this)
extracted = analyzer.extract_pdf(pdf_path)
analyzer.inspect_extraction(pdf_path, n_samples=2)

# STEP 2: Translate (comment out if don't want to translate)
translated = analyzer.translate_pdf(pdf_path)
analyzer.inspect_translation(pdf_path, n_samples=2)

✓ Loading cached extraction for 012_2015_annual_report

EXTRACTION INSPECTION: 012_2015_annual_report.pdf
Language: de
Total chunks: 12

────────────────────────────────────────────────────────────────────────────────
Chunk 1 (1909 chars)
────────────────────────────────────────────────────────────────────────────────
Verbundene Unternehmen
Inländische Unternehmen:
Saarlux Stahl GmbH & Co KG, Stuttgart T € 53,0 53,0 12 505 – 567
Dillinger Hütte Vertrieb GmbH, Stuttgart T € 100,0 100,0 4 210 1)
Ancofer Stahlhandel GmbH, Mülheim/Ruhr T € 90,0 90,0 21 278 – 3 974
Jebens GmbH, Korntal-Münchingen T € 100,0 100,0 19 808 1)
DHC-Consult GmbH, Dillingen T € 100,0 100,0 196 3
Cargo-Rail GmbH, Dillingen T € 100,0 100,0 43...

────────────────────────────────────────────────────────────────────────────────
Chunk 2 (1206 chars)
────────────────────────────────────────────────────────────────────────────────
Anlagevermögen (1)
I. Immaterielle Vermögensgegenstände 2 526 1 619
II. Sachanlagen 837 226 